# Notebook 6 — Model Training, Tuning, and Final Evaluation

This notebook trains and compares classification models using **only the features selected in Notebook 4 and prepared in Notebook 5**.

### Selected raw features
- `order_purchase_timestamp`
- `order_approved_at`
- `order_estimated_delivery_date`
- `item_count`
- `unique_products`
- `unique_sellers`
- `total_price`
- `total_freight`
- `avg_item_price`
- `payment_count`
- `total_payment_value`
- `max_installments`
- `customer_zip_code_prefix`
- `customer_state`

The datetime columns were converted into prediction-time features in Notebook 5. Post-delivery information is not used.

> **Important:** The previous version of Notebook 6 already evaluated the test set. Therefore, those earlier test results must not be used for model selection. In this notebook, all model comparison and threshold selection are done using the validation set, and the test set is evaluated only in the final evaluation section.


In [2]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
)


In [3]:
FEATURE_DIR = Path("artifacts/features")
MODEL_DIR = Path("artifacts/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_features = pd.read_csv(FEATURE_DIR / "train_features.csv")
validation_features = pd.read_csv(FEATURE_DIR / "validation_features.csv")
test_features = pd.read_csv(FEATURE_DIR / "test_features.csv")

TARGET = "is_late"

X_train = train_features.drop(columns=[TARGET])
y_train = train_features[TARGET]

X_validation = validation_features.drop(columns=[TARGET])
y_validation = validation_features[TARGET]

X_test = test_features.drop(columns=[TARGET])
y_test = test_features[TARGET]

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)


Train: (67533, 43)
Validation: (14471, 43)
Test: (14472, 43)


In [4]:
print("Training class distribution:")
print(y_train.value_counts())

print("\nTraining class percentages:")
print((y_train.value_counts(normalize=True) * 100).round(2))


Training class distribution:
is_late
0    61436
1     6097
Name: count, dtype: int64

Training class percentages:
is_late
0    90.97
1     9.03
Name: proportion, dtype: float64


## Why these models?

The data is tabular and the target is imbalanced (late orders are the minority class).

Instead of assuming that Logistic Regression or Random Forest are the right choices, this notebook compares:

1. **DummyClassifier** — simple majority baseline.
2. **HistGradientBoostingClassifier** — captures nonlinear relationships and interactions between numeric features.
3. **ExtraTreesClassifier** — another nonlinear tree ensemble that can capture interactions and complex decision boundaries.

The main model-selection metric is **Average Precision (PR-AUC)** because the positive class is relatively rare. Accuracy is reported, but it is not the main selection criterion.


In [ ]:
def evaluate_predictions(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "average_precision": average_precision_score(y_true, probabilities),}

def find_best_f1_threshold(y_true, probabilities):
    thresholds = np.linspace(0.05, 0.95, 91)
    rows = []

    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_true, predictions, zero_division=0),
            "recall": recall_score(y_true, predictions, zero_division=0),
            "f1": f1_score(y_true, predictions, zero_division=0),})

    threshold_table = pd.DataFrame(rows)
    best_row = threshold_table.loc[threshold_table["f1"].idxmax()]

    return float(best_row["threshold"]), threshold_table


## 1. Baseline model

In [6]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)

dummy_probabilities = dummy_model.predict_proba(X_validation)[:, 1]
dummy_results = evaluate_predictions(
    y_validation,
    dummy_probabilities,
    threshold=0.5
)

dummy_results


{'accuracy': 0.9465828208140419,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'average_precision': 0.053417179185958126}

## 2. Candidate model: HistGradientBoosting

In [ ]:
hist_params = [
    {"learning_rate": 0.03, "max_iter": 300, "max_leaf_nodes": 15, "min_samples_leaf": 20, "l2_regularization": 1.0},
    {"learning_rate": 0.03, "max_iter": 500, "max_leaf_nodes": 31, "min_samples_leaf": 20, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_iter": 300, "max_leaf_nodes": 31, "min_samples_leaf": 20, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_iter": 400, "max_leaf_nodes": 31, "min_samples_leaf": 20, "l2_regularization": 5.0},
    {"learning_rate": 0.05, "max_iter": 500, "max_leaf_nodes": 63, "min_samples_leaf": 20, "l2_regularization": 5.0},
    {"learning_rate": 0.08, "max_iter": 300, "max_leaf_nodes": 31, "min_samples_leaf": 20, "l2_regularization": 5.0},
    {"learning_rate": 0.10, "max_iter": 300, "max_leaf_nodes": 31, "min_samples_leaf": 20, "l2_regularization": 10.0},
    {"learning_rate": 0.05, "max_iter": 400, "max_leaf_nodes": 63, "min_samples_leaf": 30, "l2_regularization": 10.0},]

hist_tuning_rows = []

for params in hist_params:
    model = HistGradientBoostingClassifier(
        **params,
        class_weight="balanced",
        random_state=42, )
    model.fit(X_train, y_train)

    validation_probabilities = model.predict_proba(X_validation)[:, 1]
    metrics = evaluate_predictions(
        y_validation,
        validation_probabilities,
        threshold=0.5)

    hist_tuning_rows.append({
        **params,
        **metrics})

hist_tuning = pd.DataFrame(hist_tuning_rows).sort_values(
    "average_precision",
    ascending=False
).reset_index(drop=True)

hist_tuning


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,accuracy,precision,recall,f1,roc_auc,average_precision
0,0.05,500,63,20,5.0,0.850252,0.171226,0.469599,0.250951,0.744279,0.167294
1,0.08,300,31,20,5.0,0.840647,0.156124,0.450194,0.231845,0.738922,0.165261
2,0.05,400,63,30,10.0,0.851703,0.166586,0.443726,0.242232,0.736526,0.162860
3,0.05,400,31,20,5.0,0.852118,0.161130,0.420440,0.232975,0.734520,0.160718
4,0.05,300,31,20,1.0,0.848732,0.161891,0.438551,0.236484,0.734534,0.159860
5,0.10,300,31,20,10.0,0.838643,0.153197,0.446313,0.228099,0.734052,0.158177
6,0.03,300,15,20,1.0,0.838505,0.152135,0.442432,0.226415,0.736456,0.158107
7,0.03,500,31,20,1.0,0.853500,0.159677,0.408797,0.229651,0.730708,0.156841


## 3. Candidate model: ExtraTrees

In [ ]:
extra_params = [
    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 20, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 20, "min_samples_leaf": 1, "max_features": None},]

extra_tuning_rows = []

for params in extra_params:
    model = ExtraTreesClassifier(
        **params,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,)
    model.fit(X_train, y_train)

    validation_probabilities = model.predict_proba(X_validation)[:, 1]
    metrics = evaluate_predictions(
        y_validation,
        validation_probabilities,
        threshold=0.5 )

    extra_tuning_rows.append({
        **params,
        **metrics})

extra_tuning = pd.DataFrame(extra_tuning_rows).sort_values(
    "average_precision",
    ascending=False).reset_index(drop=True)

extra_tuning


,n_estimators,max_depth,min_samples_leaf,max_features,accuracy,precision,recall,f1,roc_auc,average_precision
0,300,20.0,1,NaN,0.926681,0.231343,0.160414,0.189458,0.697869,0.146273
1,300,NaN,2,sqrt,0.892129,0.179675,0.285899,0.220669,0.706531,0.144327
2,300,20.0,2,sqrt,0.815700,0.130655,0.433376,0.200779,0.695407,0.142851
3,300,12.0,2,sqrt,0.668095,0.086242,0.543338,0.148857,0.676217,0.127623


## 4. Select the final model using validation only

Average Precision is the primary metric.

The test set is **not** used here.


In [ ]:
best_hist = hist_tuning.iloc[0]
best_extra = extra_tuning.iloc[0]

candidate_summary = pd.DataFrame([
    {
        "model": "HistGradientBoosting",
        "average_precision": best_hist["average_precision"],
        "roc_auc": best_hist["roc_auc"],
        "f1_at_0.5": best_hist["f1"],},
    {
        "model": "ExtraTrees",
        "average_precision": best_extra["average_precision"],
        "roc_auc": best_extra["roc_auc"],
        "f1_at_0.5": best_extra["f1"],},
]).sort_values("average_precision", ascending=False).reset_index(drop=True)

candidate_summary


,model,average_precision,roc_auc,f1_at_0.5
0,HistGradientBoosting,0.167294,0.744279,0.250951
1,ExtraTrees,0.146273,0.697869,0.189458


In [ ]:
if candidate_summary.iloc[0]["model"] == "HistGradientBoosting":
    final_model_name = "HistGradientBoosting"
    final_params = {
        "learning_rate": float(best_hist["learning_rate"]),
        "max_iter": int(best_hist["max_iter"]),
        "max_leaf_nodes": int(best_hist["max_leaf_nodes"]),
        "l2_regularization": float(best_hist["l2_regularization"]),}

    final_model = HistGradientBoostingClassifier(
        **final_params,
        class_weight="balanced",
        random_state=42,)
else:
    final_model_name = "ExtraTrees"
    final_params = {
        "n_estimators": int(best_extra["n_estimators"]),
        "max_depth": None if pd.isna(best_extra["max_depth"]) else int(best_extra["max_depth"]),
        "min_samples_leaf": int(best_extra["min_samples_leaf"]),
        "max_features": best_extra["max_features"],}

    final_model = ExtraTreesClassifier(
        **final_params,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1, )

final_model.fit(X_train, y_train)

validation_probabilities = final_model.predict_proba(X_validation)[:, 1]

validation_ap = average_precision_score(
    y_validation,
    validation_probabilities)

validation_roc_auc = roc_auc_score(
    y_validation,
    validation_probabilities)

print("Selected model:", final_model_name)
print("Validation Average Precision:", round(validation_ap, 4))
print("Validation ROC-AUC:", round(validation_roc_auc, 4))


Selected model: HistGradientBoosting
Validation Average Precision: 0.1673
Validation ROC-AUC: 0.7443


## 5. Choose the classification threshold on validation

The default threshold of 0.50 is not automatically optimal for an imbalanced target.

We select the threshold using **validation F1**. This threshold is fixed before touching the test set.


In [ ]:
best_threshold, threshold_table = find_best_f1_threshold(
    y_validation,
    validation_probabilities)

best_threshold


0.5499999999999999

In [ ]:
validation_final_results = evaluate_predictions(
    y_validation,
    validation_probabilities,
    threshold=best_threshold)

pd.DataFrame([validation_final_results])


,accuracy,precision,recall,f1,roc_auc,average_precision
0,0.873817,0.18642,0.404916,0.255302,0.744279,0.167294


## 6. Freeze the model and evaluate the test set once

At this point:

- the model type is fixed;
- hyperparameters are fixed;
- the classification threshold is fixed;
- the test set is used only for the final evaluation.

**Do not use the test results to change the model.**


In [ ]:
# Final test evaluation — the only test evaluation in this notebook
test_probabilities = final_model.predict_proba(X_test)[:, 1]
test_results = evaluate_predictions(
    y_test,
    test_probabilities,
    threshold=best_threshold)
test_results


{'accuracy': 0.7563571033720288,
 'precision': 0.061154765971984965,
 'recall': 0.18704284221525602,
 'f1': 0.092173017507724,
 'roc_auc': 0.5747422558858128,
 'average_precision': 0.07809101084629778}

In [ ]:
test_predictions = (test_probabilities >= best_threshold).astype(int)
print(classification_report(
    y_test,
    test_predictions,
    target_names=["on_time", "late"],
    zero_division=0))


              precision    recall  f1-score   support

     on_time       0.93      0.80      0.86     13515
        late       0.06      0.19      0.09       957

    accuracy                           0.76     14472
   macro avg       0.50      0.49      0.48     14472
weighted avg       0.87      0.76      0.81     14472



## 7. Save the final model and modeling artifacts

In [ ]:
model_path = MODEL_DIR / "final_model.joblib"
joblib.dump(final_model, model_path)

model_metadata = {
    "model_name": final_model_name,
    "model_parameters": final_params,
    "selection_metric": "average_precision",
    "threshold_selection_metric": "f1",
    "selected_threshold": best_threshold,
    "training_rows": int(len(X_train)),
    "validation_rows": int(len(X_validation)),
    "test_rows": int(len(X_test)),
    "features_used": X_train.columns.tolist(),
    "test_evaluated_once_in_this_notebook": True,}

with open(MODEL_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=2)

pd.DataFrame([{
    "model": final_model_name,
    **final_params,
    "threshold": best_threshold,
    **validation_final_results
}]).to_csv(
    MODEL_DIR / "validation_final_results.csv",
    index=False)

pd.DataFrame([{
    "model": final_model_name,
    "threshold": best_threshold,
    **test_results
}]).to_csv(
    MODEL_DIR / "test_final_results.csv",
    index=False)

hist_tuning.to_csv(MODEL_DIR / "hist_gradient_boosting_tuning.csv", index=False)
extra_tuning.to_csv(MODEL_DIR / "extra_trees_tuning.csv", index=False)

print("Saved:", model_path)
print("Saved model metadata and evaluation artifacts.")


Saved: artifacts\models\final_model.joblib
Saved model metadata and evaluation artifacts.


## 8. Production inference pattern

For a new order, Notebook 5's saved preprocessing must be used first.

The production flow is:

`new raw order → same feature engineering → saved preprocessor.transform() → final_model.predict_proba() → fixed threshold`

The model and threshold are not retrained for each new order.


In [ ]:
# Example production loading pattern
loaded_model = joblib.load(MODEL_DIR / "final_model.joblib")
loaded_metadata = json.loads(
    (MODEL_DIR / "model_metadata.json").read_text(encoding="utf-8"))

print("Loaded model:", loaded_metadata["model_name"])
print("Fixed threshold:", loaded_metadata["selected_threshold"])


Loaded model: HistGradientBoosting
Fixed threshold: 0.5499999999999999


## Final conclusion

The final model is chosen from the validation results, not from the test results.

The model-selection process is:

1. Compare against a simple baseline.
2. Tune nonlinear tabular models on training data.
3. Select the best configuration using validation **Average Precision**.
4. Select the classification threshold using validation **F1**.
5. Freeze everything.
6. Evaluate the test set once at the end.
7. Save the final model, threshold, parameters, and results.
